# SFT version of 01c (built by `student/make_sft_notebooks.py`, plan in `student/SFT_PLAN.md`)

Two changes, both on by default, otherwise the notebook below is 01c unchanged:
1. **Digit clips train too** (`VERBALISE`): `vi_verbalise.py` spells the digits of the gold text the way `vi_itn.py` reads
   them back (checked over every number up to 100,000 and 20,000 random ones). This replaces the bullet below on the
   481 left-out clips: only clips with a letter+digit token (`vtv6`, `rv245`, `24h`) are still left out.
2. **Cross-fitted train transcripts** (`XFIT`): 5 more models, each trained without one fifth of the train rows for the
   best epoch of this run, transcribe the rows they never heard. `asr_xfit_train.json` is the noisy text 02 now trains on,
   and the held-out train text 03 scores on. ~15 min per fold on one T4 (estimate).

Run with `FINAL = False` (the default here) first: it prints the held-out WER per epoch and writes all files 02 and 03
need. `FINAL = True` afterwards gives the Docker's `asr_model/`; set `XFIT = False` for it, the folds would be the same.


# 01c · ASR: Gipformer 1.5 fine-tuned on the train clips, with ITN

Replaces `01b_asr_gipformer.ipynb`: same transcript files under the same names, so 02 and 03 run unchanged. Attach the output of one 01 notebook, not several.

What changes from 01b:
- Gipformer 1.5 (68.6M params, MIT, so fine-tuning is allowed) is fine-tuned on the train clips with icefall's own zipformer code on one T4, then exported to int8 ONNX in `asr_model/`. Transcribing still runs on CPU through sherpa-onnx with greedy search, as in 01b.
- Clips whose transcript has digits are left out of fine-tuning (481 of 5,036), because Gipformer only writes numbers as words and there is no verbaliser here. `vi_itn.py` turns spoken numbers into digits afterwards, as before.
- The epoch with the lowest held-out WER is kept. Epoch 0 is the stock model, so the run never gets worse than 01b.
- No PhoBERT tagger and no `ner_model/`. 03c strips the marks and 02b ver3 onwards removes them from training text, so the tagger did nothing.
- `asr_base_train.json`, the noisy text 02 trains on, comes from the **stock** model. The fine-tuned model has heard the train clips, so its transcripts of them would be cleaner than anything it produces at test time. Stock errors are a little worse than test-time errors, which is the safe direction.

Training data with `FINAL = False`: 3,529 clips, 4.65 h. With `FINAL = True`: 4,555 clips, 6.07 h. The 1,089 held-out clips are the same as 01b's, so the WER is comparable with 01b's 16.44%.

**Setup**
1. Upload `train+public_test.zip` as a **private** Kaggle dataset. The competition forbids sharing the data.
2. Settings: Accelerator **GPU T4 x2** (training uses one), Internet **on**. Add Input: that dataset.

**Run**
1. Run all with `SMOKE = True`: 200 clips, one epoch, 50 held-out clips. Note the seconds per epoch and the peak GPU memory.
2. Set `SMOKE = False`, then Save Version > Save & Run All. It prints the held-out WER of every epoch and the best one.
3. For the submission model, set `FINAL = True` and `EPOCHS` to that best epoch, then run again.

**Evaluating with 03c**: `EVAL = True` scores held-out train samples. Use this notebook's `FINAL = False` output for that. A `FINAL = True` model has heard those clips, so their `asr` score would be too good.

**Output**: `asr_model/` (fine-tuned int8 ONNX and `tokens.txt`, which the Docker needs instead of the stock files), `asr_raw.json` (no ITN), `asr_transcripts.json`, `asr_base_train.json`, `vi_itn.py`, and `exp/epoch-99.pt` (the PyTorch weights).

In [ ]:
# torch 2.8.0 with the k2 and kaldifeat builds made for it: the stack pinned in Gipformer's own pyproject.toml. Kaggle's newer torch has no k2 wheel
import sys

PY = f"cp{sys.version_info.major}{sys.version_info.minor}"
!nvidia-smi -L
!pip install -q "torch==2.8.0" "torchaudio==2.8.0" --index-url https://download.pytorch.org/whl/cu128
assert _exit_code == 0, "this step failed, see the log above"
!pip install -q "https://huggingface.co/csukuangfj/k2/resolve/main/ubuntu-cuda/k2-1.24.4.dev20250807+cuda12.8.torch2.8.0-{PY}-{PY}-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl" "https://huggingface.co/csukuangfj/kaldifeat/resolve/main/cuda/1.25.5.dev20241029/linux/kaldifeat-1.25.5.dev20250807+cuda12.8.torch2.8.0-{PY}-{PY}-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl" "sentencepiece==0.2.1" kaldialign "pypinyin==0.50.0" tensorboard jiwer "sherpa-onnx==2.0.1" onnx onnxruntime
assert _exit_code == 0, "this step failed, see the log above"
!python -c "import torch, k2, kaldifeat; print('torch', torch.__version__, torch.cuda.get_device_name(0))"
assert _exit_code == 0, "torch, k2 or kaldifeat does not load, see the log above"

In [ ]:
%%writefile lifemate.py
"""LifeMate shared code: data split, prompt format, action parser, scorer.
02 trains on this file and saves it next to the model; 03 imports that copy, so the prompt always matches the weights.
Self-check: python lifemate.py <data root>"""
import collections
import json
import sys
import unicodedata
from datetime import datetime, timedelta
from pathlib import Path

ASK = "sys.ask_missing_info"
GEN = "<|im_start|>assistant\n"
STOP = "<|im_end|>"
MAX_RESULT_CHARS = 1500  # ponytail: hard cut on long tool results, summarise per tool if truncation ever hides an ID
HOLDOUT_APPS = {"door_lock", "learning", "library", "loyalty", "subscriptions", "household_chores", "home_security"}  # train-only apps: stands in for the unseen apps of the test splits
WEEKDAYS = ["Thứ Hai", "Thứ Ba", "Thứ Tư", "Thứ Năm", "Thứ Sáu", "Thứ Bảy", "Chủ Nhật"]
RULES = """Bạn là trợ lý giọng nói tiếng Việt. Hãy xử lý yêu cầu CUỐI CÙNG của người dùng bằng cách gọi tool, người dùng có thể đổi ý giữa chừng.
Mỗi lượt trả về đúng một dòng JSON, không viết gì khác:
- Gọi tool: {"name": "<app.tool>", "arguments": {...}}
- Kết thúc: {"done": true}
Quy tắc:
- Chỉ gọi tool trong danh sách, đúng tên và đúng kiểu tham số. Không tự đoán ID, hãy đọc bằng tool trước khi ghi.
- Chỉ gọi tool thật sự cần, không đọc lại dữ liệu đã có trong hội thoại.
- Thiếu thông tin bắt buộc mà tool không tìm được: gọi sys.ask_missing_info rồi kết thúc.
- Không app nào làm được, hoặc không cần làm gì: {"done": true} ngay, không gọi tool.
- Thời gian dùng ISO-8601 đủ giây và múi giờ, ví dụ 2026-11-11T05:30:00+07:00. Số tiền là số nguyên."""


def load_split(root, split):
    with open(Path(root) / split / f"{split}.jsonl", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def load_catalog(root, split):
    with open(Path(root) / split / f"{split}_tools_catalog.json", encoding="utf-8") as f:
        return {t["name"]: t for a in json.load(f)["apps"] for t in a["tools"]}


def used_apps(x):
    return {(c["arguments"]["tool_name"] if c["name"] == ASK else c["name"]).split(".")[0] for c in x["label"]["tool_calls"]}


def clips(x):
    """Audio paths of a sample: the final request plus every user turn of the history."""
    return [x["audio"]] + [t["audio"] for t in x["history"] if t["role"] == "user"]


def gold_texts(x):
    return {x["audio"]: x["audio_text"]} | {t["audio"]: t["audio_text"] for t in x["history"] if t["role"] == "user"}


def split_rows(rows):
    """(train, val_unseen_apps, val_seen_apps). Final run: train on all rows instead."""
    train, unseen, seen = [], [], []
    for i, x in enumerate(rows):
        (unseen if used_apps(x) & HOLDOUT_APPS else seen if i % 12 == 0 else train).append(x)
    return train, unseen, seen


def render_tool(t):
    p = t["parameters"]
    o = {"name": t["name"], "description": t["description"]}
    if p.get("properties"):
        o["parameters"] = {k: {a: b for a, b in v.items() if a not in ("x-ref", "x-match")} for k, v in p["properties"].items()}
    for k in ("required", "x-one-of"):
        if p.get(k):
            o[k] = p[k]
    return json.dumps(o, ensure_ascii=False, separators=(",", ":"))


def system_prompt(ctx, tools):
    now = datetime.fromisoformat(ctx["current_time"])
    days = ", ".join(f"{WEEKDAYS[d.weekday()]} {d:%Y-%m-%d}" for d in (now + timedelta(i) for i in range(14)))
    return (f"{RULES}\n\nBây giờ: {ctx['current_time']} ({WEEKDAYS[now.weekday()]})\n14 ngày tới: {days}\n"
            f"Vị trí: {json.dumps(ctx.get('location'), ensure_ascii=False)}\n"
            f"Người dùng: {json.dumps(ctx.get('user_profile'), ensure_ascii=False)}\n\nDanh sách tool:\n" + "\n".join(render_tool(t) for t in tools))


def asst(obj):
    return {"role": "assistant", "content": json.dumps(obj, ensure_ascii=False)}


def result_msg(result):
    return {"role": "user", "content": "KẾT QUẢ: " + json.dumps(result, ensure_ascii=False)[:MAX_RESULT_CHARS]}


def build_messages(item, texts, catalog, rng=None):
    """texts: {audio path: transcript}. rng shuffles the tools, so the model cannot learn a position. Roles strictly alternate."""
    ctx = item["context"]
    names = [t for a in ctx["installed_apps"].values() for t in a["tools"]] + [ASK]
    if rng:
        rng.shuffle(names)
    msgs = [{"role": "system", "content": system_prompt(ctx, [catalog[n] for n in names])}]
    for turn in item["history"]:
        if turn["role"] == "user":
            msgs.append({"role": "user", "content": texts[turn["audio"]]})
            continue
        results = turn.get("tool_results") or []
        for i, c in enumerate(turn.get("tool_calls") or []):
            msgs += [asst({"name": c["name"], "arguments": c["arguments"]}), result_msg(results[i] if i < len(results) else {"ok": True})]
        msgs.append(asst({"done": True, "text": turn.get("text", "")}))
    msgs.append({"role": "user", "content": texts[item["audio"]]})
    return msgs


def gold_steps(x):
    """Gold trajectory after the final user turn: call, result, ..., done."""
    msgs = []
    for c, r in zip(x["label"]["tool_calls"], x["label"]["tool_results"]):
        msgs += [asst({"name": c["name"], "arguments": c["arguments"]}), result_msg(r)]
    return msgs + [asst({"done": True})]


def parse_action(text):
    """(name, arguments), or None for done / unparsable output (both end the turn)."""
    try:
        o = json.JSONDecoder().raw_decode(text[text.index("{"):])[0]
    except ValueError:
        return None
    if not isinstance(o.get("name"), str):
        return None
    return o["name"], o["arguments"] if isinstance(o.get("arguments"), dict) else {}


def tokens(s):
    return unicodedata.normalize("NFC", str(s).lower()).split()


def token_f1(a, b):
    a, b = tokens(a), tokens(b)
    c = sum((collections.Counter(a) & collections.Counter(b)).values())
    return 2 * c / (len(a) + len(b)) if c else 0.0


def call_match(p, g, catalog):
    if p["name"] != g["name"]:
        return False
    props = catalog.get(g["name"], {}).get("parameters", {}).get("properties", {})
    fuzzy = {k for k, v in props.items() if v.get("x-match") == "fuzzy"}
    pa, ga = p.get("arguments") or {}, g.get("arguments") or {}
    if {k: v for k, v in pa.items() if k not in fuzzy} != {k: v for k, v in ga.items() if k not in fuzzy}:
        return False
    return all(token_f1(pa.get(k, ""), v) > 0.6 for k, v in ga.items() if k in fuzzy)


def score(pred, gold, pred_hash, gold_hash, catalog):
    """Official formula from the description. ponytail: greedy matching, can undercount if two fuzzy calls compete for one gold call."""
    if not gold:
        return float(not pred)
    if pred_hash != gold_hash:
        return 0.0
    left, m = list(gold), 0
    for p in pred:
        i = next((i for i, g in enumerate(left) if call_match(p, g, catalog)), None)
        if i is not None:
            del left[i]
            m += 1
    if not m:
        return 0.0
    precision, recall = m / len(pred), m / len(gold)
    return 2 * precision * recall / (precision + recall)


def render(msgs):
    return "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in msgs)


def prompt_text(msgs):
    """Inference prompt. Plain ChatML instead of tokenizer.apply_chat_template: Qwen templates add a think block only to the
    last assistant turn, which would make training and inference see different text."""
    return render(msgs) + GEN


def encode(tok, msgs, first_target):
    """Tokenise one conversation, labelling only the assistant turns from msgs[first_target:]."""
    text, spans = "", []
    for i, m in enumerate(msgs):
        start = len(text) + len(f"<|im_start|>{m['role']}\n")
        text += render([m])
        if i >= first_target and m["role"] == "assistant":
            spans.append((start, len(text) - 1))  # content + <|im_end|>, without the newline
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    labels = [t if any(a <= s and e <= b for a, b in spans) else -100 for t, (s, e) in zip(enc["input_ids"], enc["offset_mapping"])]
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}


def demo(root):
    sys.path.insert(0, str(Path(root) / "train"))
    from zaic_tools import ToolEnv

    rows, catalog = load_split(root, "train"), load_catalog(root, "train")
    for x in rows[:200]:  # gold replay must score full marks, otherwise the scorer or the data path is broken
        env = ToolEnv("train", x["id"], root=root)
        gold = x["label"]["tool_calls"]
        results = [env.call(c["name"], c["arguments"]) for c in gold]
        assert results == x["label"]["tool_results"], x["id"]
        assert score(env.get_tool_calls(), gold, "h", "h", catalog) == 1.0, x["id"]

    g = [{"name": "sms.send", "arguments": {"contact_id": "c_1", "content": "mai họp lúc chín giờ"}}]
    near = [{"name": "sms.send", "arguments": {"contact_id": "c_1", "content": "Mai họp lúc chín giờ nhé"}}]
    extra = g + [{"name": "contacts.search", "arguments": {"query": "x"}}]
    wrong_id = [{"name": "sms.send", "arguments": {"contact_id": "c_2", "content": "mai họp lúc chín giờ"}}]
    assert score(near, g, "a", "a", catalog) == 1.0
    assert score(g, g, "a", "b", catalog) == 0.0
    assert score(wrong_id, g, "a", "a", catalog) == 0.0
    assert score([], g, "a", "a", catalog) == 0.0
    assert score(g, [], "a", "a", catalog) == 0.0
    assert score([], [], "x", "y", catalog) == 1.0
    assert abs(score(extra, g, "a", "a", catalog) - 2 / 3) < 1e-9

    assert parse_action('{"name": "a.b", "arguments": {"x": 1}} rác') == ("a.b", {"x": 1})
    assert parse_action('rác {"name": "a.b"}') == ("a.b", {})
    assert parse_action('{"done": true}') is None
    assert parse_action("không có json") is None
    assert parse_action('{"name": ') is None

    train, unseen, seen = split_rows(rows)
    assert len(train) + len(unseen) + len(seen) == len(rows)
    assert not any(used_apps(x) & HOLDOUT_APPS for x in train)
    for x in rows:
        msgs = build_messages(x, gold_texts(x), catalog) + gold_steps(x)
        roles = [m["role"] for m in msgs[1:]]
        assert all(a != b for a, b in zip(roles, roles[1:])), x["id"]
    print("lifemate self-check ok:", len(train), "train,", len(unseen), "unseen-app val,", len(seen), "seen-app val")


if __name__ == "__main__":
    demo(sys.argv[1])

In [ ]:
%%writefile settings.py

import glob
from pathlib import Path

SMOKE = False  # True: a short run. Then False and Save & Run All.
SMOKE_CLIPS = 200
FINAL = False  # True: fine-tune on every train clip, nothing held out
EPOCHS = 8  # FINAL: set this to the best epoch that the FINAL = False run printed. Folds use that best epoch when it exists
MAX_DURATION = 300  # seconds of padded audio per batch, lower it on OOM
BASE_LR = 0.0045  # icefall's zipformer/finetune.py default
ASR_BATCH = 32
ITN = True  # False: transcripts keep numbers as spoken
SEED = 42
VERBALISE = True  # True: clips whose gold text has digits train too, spelled out by vi_verbalise.py. False: skipped, as in 01c
XFIT = True  # True: also train XFOLDS models, each without one fold of train rows, and transcribe that fold: asr_xfit_train.json
XFOLDS = 5

ROOT = Path(glob.glob("/kaggle/input/**/train/train.jsonl", recursive=True)[0]).parents[1]
BASE_DIR = "/tmp/gipformer"  # stock Gipformer 1.5: int8 ONNX for sherpa-onnx, model.pt and bpe.model for training
ASR_DIR = "asr_model"  # the fine-tuned int8 ONNX, kept in the output for 03 and the Docker
XFIT_DIR = "/tmp/xfit"  # fold models, not kept in the output
ICEFALL = "/tmp/icefall"
ICEFALL_COMMIT = "3f848bb6d0acc970c9b294a30ca0a04a7c9c78d1"  # last zipformer change before Gipformer 1.5's release on 2026-09-17


In [ ]:
from huggingface_hub import snapshot_download

from settings import *

snapshot_download("g-group-ai-lab/gipformer1.5-68M-rnnt", revision="dd9227dcd8705c13f33bdbe59728d546ab94480f", local_dir=BASE_DIR, allow_patterns=["*.int8.onnx", "tokens.txt", "model.pt", "bpe.model"])
!rm -rf {ICEFALL} && git clone -q --depth 1 --filter=blob:none --sparse https://github.com/k2-fsa/icefall.git {ICEFALL} && cd {ICEFALL} && git sparse-checkout set icefall egs/librispeech/ASR && git fetch -q --depth 1 origin {ICEFALL_COMMIT} && git checkout -q {ICEFALL_COMMIT}
assert _exit_code == 0, "this step failed, see the log above"

Checks the data path, the scorer and the gold replay before any GPU time is spent.

In [ ]:
!python lifemate.py "{ROOT}"
assert _exit_code == 0, "this step failed, see the log above"

ITN rules, tuned on train transcripts only, unchanged from 01b. Gipformer spells every number, and gold arguments use digits. `vi_itn.py` is plain Python, so the Docker image needs only this file.

In [ ]:
%%writefile vi_itn.py
"""Inverse text normalisation for lowercase Vietnamese ASR output: spoken numbers become digits, everything else stays.
"chuyển hai trăm nghìn lúc chín rưỡi" -> "chuyển 200000 lúc 9 rưỡi". Text inside [entity marks] is left alone.
Rules come from train transcripts only. Self-check: python vi_itn.py"""
import re

DIGIT = {"không": 0, "một": 1, "hai": 2, "ba": 3, "bốn": 4, "năm": 5, "sáu": 6, "bảy": 7, "tám": 8, "chín": 9}
ONES = DIGIT | {"mốt": 1, "tư": 4, "lăm": 5}  # the last three only mean a number after another number
SCALE = {"tỷ": 10**9, "tỉ": 10**9, "triệu": 10**6, "nghìn": 1000, "ngàn": 1000}
BEFORE = {"tháng", "mùng", "mồng", "tầng", "lầu", "quận", "kênh", "tập", "mức", "kém", "số", "ngày"}  # a lone digit after these is a number
UNITS = {"giờ", "tiếng", "phút", "giây", "độ", "phần", "chỗ", "vé", "mét", "ký", "kg", "km", "lít", "đô", "đồng", "rưỡi", "ngày", "tuần", "tháng"}  # and before these
YEAR = {"nay", "sau", "ngoái", "trước"}  # "năm" as year


def at(w, i):
    return w[i] if i < len(w) else ""


def value(a):
    return DIGIT[a] if a in DIGIT else int(a) if a.isdigit() else None


def tens(w, i):
    """mười lăm, hai mươi mốt, năm chục, and the spoken short forms hai mốt, bốn lăm, hai sáu độ, hôm hai tám."""
    a, b = at(w, i), at(w, i + 1)
    if a == "mười":
        v, j = 10, i + 1
    elif DIGIT.get(a) and b in ("mươi", "chục"):
        v, j = 10 * DIGIT[a], i + 2
    elif DIGIT.get(a, 0) >= 2 and (b in ("mốt", "tư", "lăm") or (b in DIGIT and (at(w, i + 2) in ("độ", "phần") or (i and w[i - 1] in ("ngày", "hôm", "mùng", "mồng"))))):
        return 10 * DIGIT[a] + ONES[b], i + 2
    else:
        return None
    c = at(w, j)  # "năm" here is a year: mười năm, hai mươi năm. Five after a ten is spoken "lăm".
    return (v + ONES[c], j + 1) if c in ONES and c not in ("không", "năm") else (v, j)


def tail(w, j, scale):
    """hai triệu tám, hai triệu rưỡi: the unit below the scale is implied. "một" after a number means per: mười tám nghìn một ký."""
    a = at(w, j)
    if a == "rưỡi":
        return scale // 2, j + 1
    if a != "một" and ONES.get(a) and at(w, j + 1) not in UNITS | YEAR:
        return ONES[a] * scale // 10, j + 1
    return 0, j


def group(w, i):
    """One number below 1000: (value, next index), or None."""
    a, b = at(w, i), at(w, i + 1)
    if value(a) is not None and b == "trăm":
        v, j = 100 * value(a), i + 2
    elif a == "trăm" and (tens(w, i + 1) or DIGIT.get(b, 0) >= 2):  # trăm hai mốt
        v, j = 100, i + 1
    else:
        return tens(w, i) or ((value(a), i + 1) if value(a) is not None and b in SCALE else None)
    if at(w, j) in ("lẻ", "linh") and at(w, j + 1) in DIGIT:
        return v + DIGIT[w[j + 1]], j + 2
    t = tens(w, j)
    if t:
        return v + t[0], t[1]
    r, j = tail(w, j, 100)
    return v + r, j


def cardinal(w, i):
    """Longest number at w[i]: (value, next index), or None."""
    total, j, scale = 0, i, 0
    while g := group(w, j):
        v, k = g
        s = SCALE.get(at(w, k), 0)
        if not s or (scale and s >= scale):
            return (total + v, k) if not s else (total, j)
        total, j, scale = total + v * s, k + 1, s
    if not scale:
        return None
    r, j = tail(w, j, scale)
    return total + r, j


def digits(s):
    w, out, i = s.split(), [], 0
    while i < len(w):
        n = i
        while at(w, n) in DIGIT:
            n += 1
        n -= n - i > 3 and at(w, n) in UNITS  # rv hai một chín tám giờ: the last digit belongs to the unit
        if i and w[i - 1] == "thứ":  # thứ hai is Monday
            out.append(w[i])
            i += 1
        elif c := cardinal(w, i):
            out.append(str(c[0]))
            i = c[1]
        elif n - i >= 3:  # ponytail: any three digit words in a row become a digit string, "ba năm một lần" would too
            out.append("".join(str(DIGIT[x]) for x in w[i:n]))
            i = n
        elif w[i] in DIGIT and w[i] != "không" and (at(w, i - 1) in BEFORE and i or at(w, i + 1) in UNITS):
            out.append(str(DIGIT[w[i]]))
            i += 1
        else:
            out.append(w[i])
            i += 1
    return " ".join(out)


def itn(s):
    return " ".join(p if p[0] == "[" else digits(p) for p in re.split(r"(\[[^\]]*\])", s) if p.strip())


if __name__ == "__main__":
    for a, b in {
        "chuyển lan hai trăm nghìn": "chuyển lan 200000",
        "hai triệu ba trăm tám chục nghìn": "2380000",
        "một triệu rưỡi tiền cọc": "1500000 tiền cọc",
        "hai triệu tám và nạp": "2800000 và nạp",
        "hai triệu tám giờ sáng": "2000000 8 giờ sáng",
        "mười tám nghìn một ký": "18000 1 ký",
        "trăm hai mốt triệu": "121000000",
        "bốn chục triệu": "40000000",
        "năm trăm đô": "500 đô",
        "bật điều hoà hai sáu độ": "bật điều hoà 26 độ",
        "số không chín một hai bốn năm sáu bảy tám một ba": "số 09124567813",
        "chín giờ kém mười lăm": "9 giờ kém 15",
        "năm giờ bốn lăm": "5 giờ 45",
        "đặt báo thức sáu rưỡi sáng mai": "đặt báo thức 6 rưỡi sáng mai",
        "thứ hai tuần sau bay một mình": "thứ hai tuần sau bay một mình",
        "ngày mốt mở một bài": "ngày mốt mở một bài",
        "có không ba ơi năm nay": "có không ba ơi năm nay",
        "mười năm rồi": "10 năm rồi",
        "tháng chín mùng bốn": "tháng 9 mùng 4",
        "gọi [anh ba] năm phút": "gọi [anh ba] 5 phút",
        "[số một] hai mươi": "[số một] 20",
        "vài trăm nghìn": "vài trăm nghìn",
        "119 triệu": "119000000",
        "rv hai một chín tám giờ": "rv 219 8 giờ",
        "lịch hôm hai tám": "lịch hôm 28",
        "": "",
    }.items():
        assert itn(a) == b, (a, itn(a), b)
    print("vi_itn self-check ok")


In [ ]:
!python vi_itn.py
assert _exit_code == 0, "this step failed, see the log above"

Digits of the gold text spelled out for training, the inverse of `vi_itn.py`. Its self-check reads every spelled number back through `vi_itn.py`.

In [ ]:
%%writefile vi_verbalise.py
"""Spoken Vietnamese for the digits of a normalised transcript, so clips whose gold text has digits can train the ASR:
"đặt báo thức 7 giờ 30" -> "đặt báo thức bảy giờ ba mươi". The inverse of vi_itn.py, checked against it below.
A token mixing letters and digits (vtv6, rv245, 24h) gives None: how it was spoken is not known, so the clip is skipped.
Self-check: python vi_verbalise.py"""
import random

from vi_itn import digits

WORD = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]
SCALES = [(10**9, "tỷ"), (10**6, "triệu"), (1000, "nghìn")]  # train gold says nghìn 125 times, ngàn 20


def below_100(n):
    t, u = divmod(n, 10)
    if not t:
        return WORD[u]
    head = "mười" if t == 1 else f"{WORD[t]} mươi"
    if not u:
        return head
    return f"{head} " + ("lăm" if u == 5 else "mốt" if u == 1 and t > 1 else "tư" if u == 4 and t > 1 else WORD[u])


def below_1000(n, full):
    """full: a group after a higher scale keeps its hundreds, as in 'hai nghìn không trăm linh năm'."""
    h, r = divmod(n, 100)
    if not h and not full:
        return below_100(r)
    return f"{WORD[h]} trăm" + (f" linh {WORD[r]}" if 0 < r < 10 else f" {below_100(r)}" if r else "")


def cardinal(n, full=False):
    for s, name in SCALES:
        if n >= s:
            q, r = divmod(n, s)
            return f"{cardinal(q, full)} {name}" + (f" {cardinal(r, True)}" if r else "")
    return below_1000(n, full)


def by_digit(s):
    """Phone numbers, tickets, codes: a leading zero, or 6+ digits that are not a round amount."""
    return len(s) > 1 and s[0] == "0" or len(s) >= 6 and int(s) % 1000 != 0


def spoken(s):
    return " ".join(WORD[int(c)] for c in s) if by_digit(s) else cardinal(int(s))


def verbalise(text):
    out = []
    for w in text.split():
        if w.isascii() and w.isdigit():
            out.append(spoken(w))
        elif any(c.isdigit() for c in w):
            return None
        else:
            out.append(w)
    return " ".join(out)


if __name__ == "__main__":
    for a, b in {
        "đặt báo thức 7 giờ sáng mai": "đặt báo thức bảy giờ sáng mai",
        "dưới 119 triệu thì nhắc chị 9 giờ": "dưới một trăm mười chín triệu thì nhắc chị chín giờ",
        "uống thuốc lúc 9 giờ kém 15": "uống thuốc lúc chín giờ kém mười lăm",
        "bật điều hòa 24 độ": "bật điều hòa hai mươi tư độ",
        "21 25 14 10 105 2026": "hai mươi mốt hai mươi lăm mười bốn mười một trăm linh năm hai nghìn không trăm hai mươi sáu",
        "chuyển 2500000 và 2005000": "chuyển hai triệu năm trăm nghìn và hai triệu không trăm linh năm nghìn",
        "vé số 412750": "vé số bốn một hai bảy năm không",
        "gọi 0912345678": "gọi không chín một hai ba bốn năm sáu bảy tám",
        "mở kênh vtv6": None,
        "chuyến rv245": None,
        "không có số": "không có số",
        "": "",
    }.items():
        assert verbalise(a) == b, (a, verbalise(a), b)
    rng = random.Random(0)
    for n in list(range(1, 100000)) + [rng.randrange(10**10) for _ in range(20000)]:
        s, w = str(n), spoken(str(n))
        got, want = (digits(w), s) if by_digit(s) else (digits(w + " đồng"), s + " đồng")  # a unit makes vi_itn read a lone digit
        assert got == want, (s, w, got)
    print("vi_verbalise self-check ok: vi_itn reads every spelled number back")


In [ ]:
!python vi_verbalise.py
assert _exit_code == 0, "this step failed, see the log above"

In [ ]:
%%writefile asr_common.py
"""Shared by the ASR steps. Clip keys are '<split>/<audio path>'."""
import json
import os
import re
import unicodedata

import jiwer
import sherpa_onnx
import soundfile as sf

import lifemate as lm
from settings import *
from vi_itn import itn

TONES = {"̀", "́", "̃", "̉", "̣"}


def old_style_tone(word):
    """hoà -> hòa, khoẻ -> khỏe, thuỷ -> thủy: one spelling on both sides of WER."""
    d = unicodedata.normalize("NFD", word)
    tones = [c for c in d if c in TONES]
    if len(tones) != 1:
        return word
    base = unicodedata.normalize("NFC", "".join(c for c in d if c not in TONES))
    if base.endswith(("oa", "oe")) or (base.endswith("uy") and not base.endswith("quy")):
        return unicodedata.normalize("NFC", base[:-2] + base[-2] + tones[0] + base[-1])
    return word


def norm(s):
    """Lowercase, no punctuation, one tone spelling: what the model outputs, and what WER should compare."""
    s = re.sub(r"[^\w\s]|_", " ", unicodedata.normalize("NFC", s).lower())
    return " ".join(old_style_tone(w) for w in s.split())


def wer(keys, truth, hyp, f=str):
    return jiwer.wer([f(norm(truth[k])) for k in keys], [f(norm(hyp[k])) for k in keys])


def clip_texts(rows):
    return {f"train/{c}": t for x in rows for c, t in lm.gold_texts(x).items()}


def read_wav(key):
    return sf.read(str(ROOT / key), dtype="float32")[0]


def transcribe(keys, model_dir):
    """Greedy search, so deterministic. sherpa-onnx from pip runs on CPU."""
    rec = sherpa_onnx.OfflineRecognizer.from_transducer(
        encoder=f"{model_dir}/encoder.int8.onnx",
        decoder=f"{model_dir}/decoder.int8.onnx",
        joiner=f"{model_dir}/joiner.int8.onnx",
        tokens=f"{model_dir}/tokens.txt",
        num_threads=os.cpu_count(),
    )
    out = {}
    for i in range(0, len(keys), ASR_BATCH):
        streams = []
        for k in keys[i : i + ASR_BATCH]:
            streams.append(rec.create_stream())
            streams[-1].accept_waveform(16000, read_wav(k))
        rec.decode_streams(streams)
        out |= {k: norm(s.result.text) for k, s in zip(keys[i : i + ASR_BATCH], streams)}
    return out

icefall's training code imports lhotse, its data pipeline. Nothing here uses it, so it is mocked, as Gipformer's own `infer_pytorch.py` does.

In [ ]:
%%writefile ice.py
"""icefall on sys.path, with lhotse mocked as Gipformer's own infer_pytorch.py does: lhotse is icefall's data pipeline, which nothing here uses.
Import it before any icefall module. `python ice.py <zipformer script> <args>` runs that icefall script the same way."""
import importlib.abc
import importlib.util
import runpy
import sys
import types

from settings import ICEFALL


class _Anything(types.ModuleType):
    class Dummy:
        def __init__(self, *args, **kwargs):
            pass

        def __call__(self, *args, **kwargs):
            return self

        def __getattr__(self, name):
            return type(self)()

    def __getattr__(self, name):
        return self.Dummy


class _MockLhotse(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, name, path=None, target=None):
        return importlib.util.spec_from_loader(name, self, is_package=True) if name.split(".")[0] == "lhotse" else None

    def create_module(self, spec):
        module = _Anything(spec.name)
        module.__path__, module.__version__, module.__file__ = [], "0.0.0", "<mocked>"  # icefall's get_env_info() reads lhotse.__file__
        return module

    def exec_module(self, module):
        pass


sys.meta_path.insert(0, _MockLhotse())
ZIPFORMER = f"{ICEFALL}/egs/librispeech/ASR/zipformer"
sys.path[:0] = [ICEFALL, f"{ICEFALL}/egs/librispeech/ASR", ZIPFORMER]

if __name__ == "__main__":
    sys.argv = sys.argv[1:]
    runpy.run_path(f"{ZIPFORMER}/{sys.argv[0]}", run_name="__main__")


Fine-tune on one T4 with icefall's settings for fine-tuning a converged zipformer: ScaledAdam at 0.0045, pruned RNN-T loss, SpecAugment, fp16.

In [ ]:
%%writefile train_asr.py
"""Fine-tune Gipformer 1.5 on the train clips with icefall's own model code, on one GPU. Keeps the epoch with the lowest held-out WER
(the last one when FINAL) and saves it as exp/epoch-99.pt for icefall's export-onnx.py."""
import argparse
import math
import random
import sys
import time

import ice  # noqa: F401, before any icefall import

import k2
import kaldifeat
import sentencepiece as spm
import torch
import torchaudio
from beam_search import greedy_search_batch
from optim import Eden, ScaledAdam
from train import add_model_arguments, get_model, get_params, set_batch_count

from icefall.utils import create_grad_scaler, get_parameter_groups_with_lrs, num_tokens, torch_autocast

from asr_common import *
from vi_verbalise import verbalise

FOLD = int(sys.argv[1]) if len(sys.argv) > 1 else None  # cross-fit: train without the rows i % XFOLDS == FOLD

random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FP16 = DEVICE.type == "cuda"

params = get_params()
parser = argparse.ArgumentParser()
add_model_arguments(parser)
params.update(vars(parser.parse_args([])))  # Gipformer is icefall's default zipformer, as in its infer_pytorch.py
symbols = k2.SymbolTable.from_file(f"{BASE_DIR}/tokens.txt")
params.update(blank_id=symbols["<blk>"], vocab_size=num_tokens(symbols) + 1, context_size=2)
model = get_model(params)
state = torch.load(f"{BASE_DIR}/model.pt", map_location="cpu", weights_only=False)
missing, unused = model.load_state_dict(state.get("model", state), strict=False)
assert not missing, f"the checkpoint does not fit icefall's default zipformer: {missing[:5]}"  # Gipformer's script loads non-strict, which would hide this
print(f"{sum(p.numel() for p in model.parameters()):,} params, checkpoint keys not used: {unused}")
model.to(DEVICE)
set_batch_count(model, 100000)  # as finetune.py: a converged model keeps its scheduled dropouts and balancers at their final values
sp = spm.SentencePieceProcessor(model_file=f"{BASE_DIR}/bpe.model")
unk = sp.piece_to_id("<unk>")

opts = kaldifeat.FbankOptions()  # the features of Gipformer's infer_pytorch.py
opts.device = DEVICE
opts.frame_opts.dither = 0
opts.frame_opts.snip_edges = False
opts.frame_opts.samp_freq = 16000
opts.mel_opts.num_bins = 80
opts.mel_opts.high_freq = -400
fbank = kaldifeat.Fbank(opts)

rows = lm.load_split(ROOT, "train")
if FOLD is None:
    fit, *held = (rows, [], []) if FINAL else lm.split_rows(rows)
else:
    fit, held = [x for i, x in enumerate(rows) if i % XFOLDS != FOLD], []
truth = clip_texts(rows)
held_keys = list(clip_texts(sum(held, [])))[: 50 if SMOKE else None]
data, skipped = [], 0
for k, text in clip_texts(fit).items():
    spoken = verbalise(norm(text)) if VERBALISE else None if re.search(r"\d", text) else norm(text)  # None: digits of unknown spoken form
    y = sp.encode(spoken.upper()) if spoken else [unk]  # the model's own style: uppercase, old tone spelling, numbers as words
    if unk in y:
        skipped += 1
    else:
        data.append((k, y))
data = data[:SMOKE_CLIPS] if SMOKE else data
feats = {k: fbank(torch.from_numpy(read_wav(k)).to(DEVICE)).cpu() for k in [k for k, _ in data] + held_keys}
spelled = sum(bool(re.search(r"\d", truth[k])) for k, _ in data)
print(f"{len(data)} training clips ({spelled} with digits spelled out), {sum(len(feats[k]) for k, _ in data) / 360000:.1f} h, {skipped} skipped for unknown digits or tokens, {len(held_keys)} held-out clips, fold {FOLD}")


def batches(items):
    """Sorted by length, cut where the padded batch passes MAX_DURATION (100 frames per second)."""
    out, cur = [], []
    for item in sorted(items, key=lambda x: len(feats[x[0]])):
        if cur and (len(cur) + 1) * len(feats[item[0]]) > MAX_DURATION * 100:
            out.append(cur)
            cur = []
        cur.append(item)
    return out + [cur]


FREQ_MASK, TIME_MASK = torchaudio.transforms.FrequencyMasking(27), torchaudio.transforms.TimeMasking(100, p=0.015)


def spec_augment(f):
    """Roughly icefall's default: 2 frequency masks of up to 27 bins, 10 time masks of up to 1.5% each, filled with the mean."""
    x = f.T.unsqueeze(0)
    for mask in [FREQ_MASK] * 2 + [TIME_MASK] * 10:
        x = mask(x, mask_value=float(x.mean()))
    return x[0].T


def collate(batch, augment):
    xs = [spec_augment(feats[k]) if augment else feats[k] for k, _ in batch]
    x = torch.nn.utils.rnn.pad_sequence(xs, batch_first=True, padding_value=math.log(1e-10)).to(DEVICE)
    return x, torch.tensor([len(f) for f in xs], device=DEVICE)


@torch.inference_mode()
def held_out_wer():
    model.eval()
    hyp = {}
    for b in batches([(k, None) for k in held_keys]):
        enc, enc_lens = model.forward_encoder(*collate(b, False))
        hyp |= {k: norm(sp.decode(h)) for (k, _), h in zip(b, greedy_search_batch(model=model, encoder_out=enc, encoder_out_lens=enc_lens))}
    model.train()
    return wer(held_keys, truth, hyp)


optimizer = ScaledAdam(get_parameter_groups_with_lrs(model, lr=BASE_LR, include_names=True), lr=BASE_LR, clipping_scale=2.0)
scheduler = Eden(optimizer, 100000, 100)  # finetune.py's lr_batches and lr_epochs: flat after Eden's 500-step warmup
scaler = create_grad_scaler(enabled=FP16, init_scale=1.0)
best = (held_out_wer(), 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}) if held_keys else None
if best:
    print(f"epoch 0 (stock, PyTorch greedy) held-out WER {best[0]:.4f}", flush=True)
model.train()
step = 0
epochs = int(Path("exp/best_epoch").read_text()) if FOLD is not None and Path("exp/best_epoch").exists() else EPOCHS  # the main run's best
for epoch in range(1, (1 if SMOKE else epochs) + 1):
    start, losses, frames = time.time(), 0.0, 0
    order = batches(data)
    random.shuffle(order)
    for b in order:
        step += 1
        x, x_lens = collate(b, True)
        with torch_autocast(enabled=FP16):
            simple, pruned, *_ = model(x=x, x_lens=x_lens, y=k2.RaggedTensor([y for _, y in b]), prune_range=5, am_scale=0.0, lm_scale=0.25)
            loss = 0.5 * simple + pruned  # finetune.py's loss once its warm-up is over: the model is already trained
        scaler.scale(loss).backward()
        scheduler.step_batch(step)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        if FP16 and step % 100 == 0 and scaler.get_scale() < 8.0:  # icefall's own recovery from a collapsed fp16 scale
            scaler.update(scaler.get_scale() * 2.0)
        losses, frames = losses + loss.item(), frames + int(x_lens.sum()) // 4
    scheduler.step_epoch(epoch)
    line = f"epoch {epoch}: {len(order)} batches, loss per frame {losses / max(frames, 1):.4f}, {time.time() - start:.0f} s"
    if FP16:
        line += f", peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB, fp16 scale {scaler.get_scale()}"
    if held_keys:
        w = held_out_wer()
        line += f", held-out WER {w:.4f}"
        if w < best[0]:
            best = (w, epoch, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
    print(line, flush=True)

if best:
    print(f"best epoch {best[1]}, held-out WER {best[0]:.4f}. For FINAL = True set EPOCHS = {best[1]}")
out = Path("exp" if FOLD is None else f"{XFIT_DIR}/fold{FOLD}")
out.mkdir(parents=True, exist_ok=True)
torch.save({"model": best[2] if best else model.state_dict()}, out / "epoch-99.pt")
if best:
    Path("exp/best_epoch").write_text(str(best[1]))


In [ ]:
!python -u train_asr.py
assert _exit_code == 0, "this step failed, see the log above"

Export with icefall's own `export-onnx.py` and int8 quantisation, the format 01b used from the hub.

In [ ]:
import shutil

from settings import *

!python ice.py export-onnx.py --tokens {BASE_DIR}/tokens.txt --epoch 99 --avg 1 --use-averaged-model 0 --exp-dir exp
assert _exit_code == 0, "this step failed, see the log above"
Path(ASR_DIR).mkdir(exist_ok=True)
for part in ("encoder", "decoder", "joiner"):
    shutil.copy(f"exp/{part}-epoch-99-avg-1.int8.onnx", f"{ASR_DIR}/{part}.int8.onnx")
shutil.copy(f"{BASE_DIR}/tokens.txt", ASR_DIR)
for f in Path("exp").glob("*-avg-1.onnx"):  # the float32 exports, about 270 MB, unused
    f.unlink()
!ls -la {ASR_DIR} exp

Transcribe every clip with the fine-tuned model, and the train clips again with the stock one. Both run on CPU. Prints the held-out WER of both.

In [ ]:
%%writefile asr_transcribe.py
"""Transcribe every clip with the fine-tuned model for 03, and the train clips with the stock model for 02.

The fine-tuned model has heard the train clips, so its transcripts of them are too clean to teach an LLM what test-time errors look like.
02 gets the stock model's instead: noisier than test, which is the safe direction."""
from asr_common import *

rows = lm.load_split(ROOT, "train")
truth = clip_texts(rows)
keys = list(truth) + [f"public_test/{c}" for x in lm.load_split(ROOT, "public_test") for c in lm.clips(x)]
keys = keys[:SMOKE_CLIPS] if SMOKE else keys
tuned = transcribe(keys, ASR_DIR)
stock = transcribe([k for k in keys if k.startswith("train/")], BASE_DIR)
fix = itn if ITN else str
json.dump(tuned, open("asr_raw.json", "w", encoding="utf-8"), ensure_ascii=False)
json.dump({k: fix(v) for k, v in tuned.items()}, open("asr_transcripts.json", "w", encoding="utf-8"), ensure_ascii=False)
json.dump({k.split("/", 1)[1]: fix(v) for k, v in stock.items()}, open("asr_base_train.json", "w", encoding="utf-8"), ensure_ascii=False)

val_keys = [k for k in clip_texts(sum(lm.split_rows(rows)[1:], [])) if k in tuned]
for name, hyp in (("stock", stock), ("fine-tuned" + (", trained on these clips" if FINAL else ""), tuned)):
    print(f"{name}: WER on {len(val_keys)} held-out clips {wer(val_keys, truth, hyp):.4f}, after ITN on both sides {wer(val_keys, truth, hyp, itn):.4f}")
for k in val_keys[:5]:
    print("REF  ", truth[k], "\nSTOCK", fix(stock[k]), "\nTUNED", fix(tuned[k]))
print(len(tuned), "transcripts written")


In [ ]:
!python asr_transcribe.py
assert _exit_code == 0, "this step failed, see the log above"

Cross-fitted train transcripts for 02 and 03 (`XFIT`): one model per fold of train rows, each transcribing only the fold it never heard.

In [ ]:
%%writefile asr_xfit.py
"""Cross-fitted transcripts of the train clips for 02: every clip is transcribed by the fold model that never heard it, so the
LLM learns from errors like the ones the fine-tuned model makes at test time (the stock model's are a different kind).
Same keys and ITN as asr_base_train.json. Folds are by train row, so a row's history and final clips share one."""
from asr_common import *

rows = lm.load_split(ROOT, "train")
truth = clip_texts(rows)
out = {}
for f in range(1 if SMOKE else XFOLDS):
    keys = list(clip_texts([x for i, x in enumerate(rows) if i % XFOLDS == f]))[: SMOKE_CLIPS if SMOKE else None]
    hyp = transcribe(keys, f"{XFIT_DIR}/fold{f}")
    print(f"fold {f}: {len(keys)} clips, WER {wer(keys, truth, hyp):.4f}, after ITN on both sides {wer(keys, truth, hyp, itn):.4f}", flush=True)
    out |= hyp
assert SMOKE or out.keys() == truth.keys(), "some train clips have no cross-fitted transcript"
val = [k for k in clip_texts(sum(lm.split_rows(rows)[1:], [])) if k in out]
print(f"{len(out)} clips, WER after ITN {wer(list(out), truth, out, itn):.4f}; on lifemate's {len(val)} held-out clips {wer(val, truth, out, itn):.4f} (stock 0.1609)")
fix = itn if ITN else str
json.dump({k.split("/", 1)[1]: fix(v) for k, v in out.items()}, open("asr_xfit_train.json", "w", encoding="utf-8"), ensure_ascii=False)


In [ ]:
import shutil

from settings import *

if XFIT:  # SMOKE: fold 0 only, 1 epoch, 200 clips, to prove the path
    for f in range(1 if SMOKE else XFOLDS):
        d = f"{XFIT_DIR}/fold{f}"
        !python -u train_asr.py {f}
        assert _exit_code == 0, f"fold {f} training failed, see the log above"
        !python ice.py export-onnx.py --tokens {BASE_DIR}/tokens.txt --epoch 99 --avg 1 --use-averaged-model 0 --exp-dir {d}
        assert _exit_code == 0, f"fold {f} export failed, see the log above"
        for part in ("encoder", "decoder", "joiner"):
            shutil.move(f"{d}/{part}-epoch-99-avg-1.int8.onnx", f"{d}/{part}.int8.onnx")
        shutil.copy(f"{BASE_DIR}/tokens.txt", d)
        for p in [*Path(d).glob("*-avg-1.onnx"), Path(d, "epoch-99.pt")]:  # /tmp, but a fold is ~600 MB
            p.unlink()
    !python asr_xfit.py
    assert _exit_code == 0, "this step failed, see the log above"
